In [ ]:
rdm_url = None
idp_name_1 = None
idp_username_1 = None
idp_password_1 = None
rdm_project_name = None
target_storage_name = 'NII Storage'
delete_project = False
default_result_path = None
close_on_fail = False
transition_timeout = 60000
render_timeout = 120000

# GakuNinRDM 総合テスト [ストレージ共通-ファイルプレビュー]

- サブシステム名: アドオン
- ページ/アドオン: ストレージのファイル画面
- 機能分類: ファイルプレビュー
- シナリオ名: 形式別の初回表示・操作・再表示
- 用意するテストデータ: `testdata/mfr-preview/` の9ファイル、接続先URL、既存ユーザーのアカウント


In [ ]:
from datetime import datetime
from getpass import getpass
from pathlib import Path
from uuid import uuid4
import importlib
import tempfile

if rdm_url is None:
    rdm_url = input('GRDM URL: ')
if idp_username_1 is None:
    idp_username_1 = input(f'Username for {idp_name_1}: ')
if idp_password_1 is None and idp_name_1 != 'FakeCAS':
    idp_password_1 = getpass(f'Password for {idp_username_1}@{idp_name_1}: ')
if rdm_project_name is None:
    rdm_project_name = f'TEST-MFR-{datetime.now():%Y%m%d-%H%M%S}-{uuid4().hex[:8]}'
if default_result_path is None:
    default_result_path = tempfile.mkdtemp(prefix='mfr-preview-')

fixtures = Path('testdata/mfr-preview').resolve()
for name in ('preview.txt', 'preview.png', 'preview.jpg', 'preview.pdf',
             'preview.docx', 'preview.pptx', 'preview.csv', 'preview.xlsx', 'preview.webm'):
    assert (fixtures / name).is_file(), name
print(f'Project: {rdm_project_name}')
print(f'Results: {default_result_path}')

import scripts.playwright
importlib.reload(scripts.playwright)
from scripts.playwright import init_pw_context, run_pw, finish_pw_context, expect
from scripts import grdm
from scripts import mfr_preview as preview

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)

## 新規プライベートウィンドウで GRDM トップページを開く

設定した URL を開く。同意画面が出た場合は内容を確認して「同意する」を押す。ログイン画面が表示されること。

In [ ]:
async def _step(page):
    await page.set_viewport_size({'width': 1440, 'height': 1000})
    await page.goto(rdm_url)
    consent = page.locator('//button[text()="同意する"]')
    if await consent.count():
        await consent.click()
    await grdm.expect_anonymous_toppage(page, idp_name_1, transition_timeout=transition_timeout)

await run_pw(_step)

## 指定したアカウントでログインする

設定した IdP を選択し、ユーザー名とパスワードを入力してログインする。プロジェクト一覧が表示されること。

In [ ]:
async def _step(page):
    await grdm.login(page, idp_name_1, idp_username_1, idp_password_1,
                     transition_timeout=transition_timeout)
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)

## テスト専用の新規プロジェクトを作成する

「新規プロジェクト作成」を押し、上に表示したプロジェクト名を入力して作成する。作成後は「ここに留まる」を選ぶ。同名の既存プロジェクトがある場合は、別の名前で始め直す。

In [ ]:
async def _step(page):
    await expect(page.locator('[data-test-dashboard-item-title]').filter(
        has_text=rdm_project_name
    )).to_have_count(0)
    await page.locator('[data-test-create-project-modal-button]').click()
    await page.locator('input.project-name').fill(rdm_project_name)
    await page.locator('[data-test-create-project-submit]').click()
    await page.locator('button[data-test-stay-here]').click()
    await expect(page.get_by_text(rdm_project_name, exact=True)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 作成したプロジェクトを開き、上部の「ファイル」をクリックする

対象ストレージが展開され、空のファイル一覧が表示されること。

In [ ]:
async def _step(page):
    await page.get_by_text(rdm_project_name, exact=True).click()
    await expect(page.locator('//a[text()="アドオン"]')).to_be_visible(timeout=transition_timeout)
    await page.locator('#projectNavFiles a').click()
    await expect(grdm.get_select_expanded_storage_title_locator(
        page, target_storage_name
    )).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 対象ストレージに preview.txt をアップロードする

ストレージ名をクリックし、「アップロード」で `testdata/mfr-preview/preview.txt` を選択する。アップロードが完了し、一覧に `preview.txt` が表示されること。

In [ ]:
async def _step(page):
    await grdm.get_select_storage_title_locator(page, target_storage_name).click()
    await grdm.upload_file(page, str(fixtures / 'preview.txt'))
    await grdm.wait_for_uploaded(page, 'preview.txt', timeout=transition_timeout)

await run_pw(_step)

## 対象ストレージに preview.png をアップロードする

ストレージ名をクリックし、「アップロード」で `testdata/mfr-preview/preview.png` を選択する。アップロードが完了し、一覧に `preview.png` が表示されること。

In [ ]:
async def _step(page):
    await grdm.get_select_storage_title_locator(page, target_storage_name).click()
    await grdm.upload_file(page, str(fixtures / 'preview.png'))
    await grdm.wait_for_uploaded(page, 'preview.png', timeout=transition_timeout)

await run_pw(_step)

## 対象ストレージに preview.jpg をアップロードする

ストレージ名をクリックし、「アップロード」で `testdata/mfr-preview/preview.jpg` を選択する。アップロードが完了し、一覧に `preview.jpg` が表示されること。

In [ ]:
async def _step(page):
    await grdm.get_select_storage_title_locator(page, target_storage_name).click()
    await grdm.upload_file(page, str(fixtures / 'preview.jpg'))
    await grdm.wait_for_uploaded(page, 'preview.jpg', timeout=transition_timeout)

await run_pw(_step)

## 対象ストレージに preview.pdf をアップロードする

ストレージ名をクリックし、「アップロード」で `testdata/mfr-preview/preview.pdf` を選択する。アップロードが完了し、一覧に `preview.pdf` が表示されること。

In [ ]:
async def _step(page):
    await grdm.get_select_storage_title_locator(page, target_storage_name).click()
    await grdm.upload_file(page, str(fixtures / 'preview.pdf'))
    await grdm.wait_for_uploaded(page, 'preview.pdf', timeout=transition_timeout)

await run_pw(_step)

## 対象ストレージに preview.docx をアップロードする

ストレージ名をクリックし、「アップロード」で `testdata/mfr-preview/preview.docx` を選択する。アップロードが完了し、一覧に `preview.docx` が表示されること。

In [ ]:
async def _step(page):
    await grdm.get_select_storage_title_locator(page, target_storage_name).click()
    await grdm.upload_file(page, str(fixtures / 'preview.docx'))
    await grdm.wait_for_uploaded(page, 'preview.docx', timeout=transition_timeout)

await run_pw(_step)

## 対象ストレージに preview.pptx をアップロードする

ストレージ名をクリックし、「アップロード」で `testdata/mfr-preview/preview.pptx` を選択する。アップロードが完了し、一覧に `preview.pptx` が表示されること。

In [ ]:
async def _step(page):
    await grdm.get_select_storage_title_locator(page, target_storage_name).click()
    await grdm.upload_file(page, str(fixtures / 'preview.pptx'))
    await grdm.wait_for_uploaded(page, 'preview.pptx', timeout=transition_timeout)

await run_pw(_step)

## 対象ストレージに preview.csv をアップロードする

ストレージ名をクリックし、「アップロード」で `testdata/mfr-preview/preview.csv` を選択する。アップロードが完了し、一覧に `preview.csv` が表示されること。

In [ ]:
async def _step(page):
    await grdm.get_select_storage_title_locator(page, target_storage_name).click()
    await grdm.upload_file(page, str(fixtures / 'preview.csv'))
    await grdm.wait_for_uploaded(page, 'preview.csv', timeout=transition_timeout)

await run_pw(_step)

## 対象ストレージに preview.xlsx をアップロードする

ストレージ名をクリックし、「アップロード」で `testdata/mfr-preview/preview.xlsx` を選択する。アップロードが完了し、一覧に `preview.xlsx` が表示されること。

In [ ]:
async def _step(page):
    await grdm.get_select_storage_title_locator(page, target_storage_name).click()
    await grdm.upload_file(page, str(fixtures / 'preview.xlsx'))
    await grdm.wait_for_uploaded(page, 'preview.xlsx', timeout=transition_timeout)

await run_pw(_step)

## 対象ストレージに preview.webm をアップロードする

ストレージ名をクリックし、「アップロード」で `testdata/mfr-preview/preview.webm` を選択する。アップロードが完了し、一覧に `preview.webm` が表示されること。

In [ ]:
async def _step(page):
    await grdm.get_select_storage_title_locator(page, target_storage_name).click()
    await grdm.upload_file(page, str(fixtures / 'preview.webm'))
    await grdm.wait_for_uploaded(page, 'preview.webm', timeout=transition_timeout)

await run_pw(_step)

## 一覧から preview.txt を開く

「MFR preview test」「日本語の表示確認：研究データ」「ASCII: ABC xyz 0123456789」の3行が表示されること。日本語が欠けたり文字化けしたりせず、改行が保たれていること。

In [ ]:
async def _step(page):
    await preview.open_file(page, 'preview.txt', render_timeout)
    await expect(preview.viewer(page).locator('pre')).to_have_text(
        (fixtures / 'preview.txt').read_text(encoding='utf-8'), timeout=render_timeout
    )

await run_pw(_step)

## 上部の「ファイル」をクリックして一覧に戻る

プレビューが閉じ、対象ストレージのファイル一覧が表示されること。

In [ ]:
async def _step(page):
    await preview.file_list(page, target_storage_name, transition_timeout)

await run_pw(_step)

## 一覧から preview.txt をもう一度開く

「MFR preview test」「日本語の表示確認：研究データ」「ASCII: ABC xyz 0123456789」の3行が表示されること。日本語が欠けたり文字化けしたりせず、改行が保たれていること。

In [ ]:
async def _step(page):
    await preview.open_file(page, 'preview.txt', render_timeout)
    await expect(preview.viewer(page).locator('pre')).to_have_text(
        (fixtures / 'preview.txt').read_text(encoding='utf-8'), timeout=render_timeout
    )

await run_pw(_step)

## 上部の「ファイル」をクリックして一覧に戻る

プレビューが閉じ、対象ストレージのファイル一覧が表示されること。

In [ ]:
async def _step(page):
    await preview.file_list(page, target_storage_name, transition_timeout)

await run_pw(_step)

## 一覧から preview.png を開く

左上に赤い長方形、右上に青い円、左下に緑の三角形、右下に「MFR 0123456789」が表示されること。欠けや縦横比の崩れがないこと。

In [ ]:
async def _step(page):
    await preview.open_file(page, 'preview.png', render_timeout)
    await preview.expect_image(page, render_timeout)

await run_pw(_step)

## preview.png の画像をクリックし、ホイールを上に回して拡大する

画像の中央をクリックして拡大表示を有効にし、画像上でマウスホイールを上に1回回す。図形と文字がさらに大きく表示されること。

In [ ]:
async def _step(page):
    global zoom_original_width
    frame = preview.viewer(page)
    await expect(frame.locator('.zoomImage')).to_have_count(1, timeout=render_timeout)
    await frame.locator('#base-image').locator('..').click()
    zoom = frame.locator('.zoomImage')
    await expect(zoom).to_have_css('opacity', '1')
    zoom_original_width = await zoom.evaluate('image => image.width')
    await page.mouse.wheel(0, -100)
    await preview.wait_image_width(page, zoom_original_width, larger=True, timeout=render_timeout)

await run_pw(_step)

## preview.png の画像上でホイールを下に回して縮小する

画像上でマウスホイールを下に1回回す。図形と文字が拡大前の大きさに戻ること。

In [ ]:
async def _step(page):
    await preview.viewer(page).locator('.zoomImage').hover()
    await page.mouse.wheel(0, 100)
    await preview.wait_image_width(page, zoom_original_width, larger=False, timeout=render_timeout)

await run_pw(_step)

## 上部の「ファイル」をクリックして一覧に戻る

プレビューが閉じ、対象ストレージのファイル一覧が表示されること。

In [ ]:
async def _step(page):
    await preview.file_list(page, target_storage_name, transition_timeout)

await run_pw(_step)

## 一覧から preview.png をもう一度開く

左上に赤い長方形、右上に青い円、左下に緑の三角形、右下に「MFR 0123456789」が表示されること。欠けや縦横比の崩れがないこと。

In [ ]:
async def _step(page):
    await preview.open_file(page, 'preview.png', render_timeout)
    await preview.expect_image(page, render_timeout)

await run_pw(_step)

## 上部の「ファイル」をクリックして一覧に戻る

プレビューが閉じ、対象ストレージのファイル一覧が表示されること。

In [ ]:
async def _step(page):
    await preview.file_list(page, target_storage_name, transition_timeout)

await run_pw(_step)

## 一覧から preview.jpg を開く

左上に赤い長方形、右上に青い円、左下に緑の三角形、右下に「MFR 0123456789」が表示されること。欠けや縦横比の崩れがないこと。

In [ ]:
async def _step(page):
    await preview.open_file(page, 'preview.jpg', render_timeout)
    await preview.expect_image(page, render_timeout)

await run_pw(_step)

## preview.jpg の画像をクリックし、ホイールを上に回して拡大する

画像の中央をクリックして拡大表示を有効にし、画像上でマウスホイールを上に1回回す。図形と文字がさらに大きく表示されること。

In [ ]:
async def _step(page):
    global zoom_original_width
    frame = preview.viewer(page)
    await expect(frame.locator('.zoomImage')).to_have_count(1, timeout=render_timeout)
    await frame.locator('#base-image').locator('..').click()
    zoom = frame.locator('.zoomImage')
    await expect(zoom).to_have_css('opacity', '1')
    zoom_original_width = await zoom.evaluate('image => image.width')
    await page.mouse.wheel(0, -100)
    await preview.wait_image_width(page, zoom_original_width, larger=True, timeout=render_timeout)

await run_pw(_step)

## preview.jpg の画像上でホイールを下に回して縮小する

画像上でマウスホイールを下に1回回す。図形と文字が拡大前の大きさに戻ること。

In [ ]:
async def _step(page):
    await preview.viewer(page).locator('.zoomImage').hover()
    await page.mouse.wheel(0, 100)
    await preview.wait_image_width(page, zoom_original_width, larger=False, timeout=render_timeout)

await run_pw(_step)

## 上部の「ファイル」をクリックして一覧に戻る

プレビューが閉じ、対象ストレージのファイル一覧が表示されること。

In [ ]:
async def _step(page):
    await preview.file_list(page, target_storage_name, transition_timeout)

await run_pw(_step)

## 一覧から preview.jpg をもう一度開く

左上に赤い長方形、右上に青い円、左下に緑の三角形、右下に「MFR 0123456789」が表示されること。欠けや縦横比の崩れがないこと。

In [ ]:
async def _step(page):
    await preview.open_file(page, 'preview.jpg', render_timeout)
    await preview.expect_image(page, render_timeout)

await run_pw(_step)

## 上部の「ファイル」をクリックして一覧に戻る

プレビューが閉じ、対象ストレージのファイル一覧が表示されること。

In [ ]:
async def _step(page):
    await preview.file_list(page, target_storage_name, transition_timeout)

await run_pw(_step)

## 一覧から preview.pdf を開く

全2ページのうち1ページ目が表示されること。「日本語プレビュー 1ページ目」「研究データ」「MFR PDF page 1」と、画像と同じ3色の図形・英数字が読めること。日本語が空白や四角になっていないこと。

In [ ]:
async def _step(page):
    await preview.open_file(page, 'preview.pdf', render_timeout)
    await preview.expect_pdf_page(page, 1, '研究データ', render_timeout)

await run_pw(_step)

## PDF の「次のページ」を押す

ページ番号が2に変わり、「日本語プレビュー 2ページ目」「検索確認」「MFR PDF page 2」と図形が表示されること。

In [ ]:
async def _step(page):
    await preview.viewer(page).locator('#next').click()
    await preview.expect_pdf_page(page, 2, '検索確認', render_timeout)

await run_pw(_step)

## PDF の「前のページ」を押す

全2ページのうち1ページ目が表示されること。「日本語プレビュー 1ページ目」「研究データ」「MFR PDF page 1」と、画像と同じ3色の図形・英数字が読めること。日本語が空白や四角になっていないこと。

In [ ]:
async def _step(page):
    await preview.viewer(page).locator('#previous').click()
    await preview.expect_pdf_page(page, 1, '研究データ', render_timeout)

await run_pw(_step)

## PDF の検索欄で「検索確認」を検索する

ツールバーの検索ボタンを押し、「検索確認」を入力して Enter を押す。2ページ目に移動し、一致した文字が強調表示されること。

In [ ]:
async def _step(page):
    frame = preview.viewer(page)
    await frame.locator('#viewFind').click()
    await frame.locator('#findInput').fill('検索確認')
    await frame.locator('#findInput').press('Enter')
    await preview.expect_pdf_page(page, 2, '検索確認', render_timeout)
    await expect(frame.locator('.textLayer .highlight.selected')).to_have_text('検索確認', timeout=render_timeout)

await run_pw(_step)

## PDF の「印刷」を選ぶ

ツールバーの「印刷」を押す。画面幅が狭く印刷ボタンが隠れている場合は、右端の追加ツールボタン（≫）を開き、「印刷」を押す。印刷ダイアログに2ページが現れ、日本語と図形が含まれることを確認し、キャンセルする。

コードは実際の印刷ボタンを押し、2ページ分の印刷用 canvas が作られ、印刷処理が終了することを確認する。
コードで使用するヘッドレスブラウザーには印刷ダイアログが出ないため、ダイアログ内のページ数・字形・配置は人が別途確認する。

In [ ]:
async def _step(page):
    await preview.observe_pdf_print(page)
    frame = preview.viewer(page)
    if await frame.locator('#print').is_visible():
        await frame.locator('#print').click()
    else:
        await frame.locator('#secondaryToolbarToggle').click()
        await frame.locator('#secondaryPrint').click()
    await preview.expect_pdf_print(page, render_timeout)

await run_pw(_step)

## 上部の「ファイル」をクリックして一覧に戻る

プレビューが閉じ、対象ストレージのファイル一覧が表示されること。

In [ ]:
async def _step(page):
    await preview.file_list(page, target_storage_name, transition_timeout)

await run_pw(_step)

## 一覧から preview.pdf をもう一度開く

ページ番号欄に `1` を入力して Enter を押す。前回の表示位置にかかわらず、1ページ目を確認する。

全2ページのうち1ページ目が表示されること。「日本語プレビュー 1ページ目」「研究データ」「MFR PDF page 1」と、画像と同じ3色の図形・英数字が読めること。日本語が空白や四角になっていないこと。

In [ ]:
async def _step(page):
    await preview.open_file(page, 'preview.pdf', render_timeout)
    await expect(preview.viewer(page).locator('#viewer .page canvas').first).to_be_visible(timeout=render_timeout)
    await preview.viewer(page).locator('#pageNumber').fill('1')
    await preview.viewer(page).locator('#pageNumber').press('Enter')
    await preview.expect_pdf_page(page, 1, '研究データ', render_timeout)

await run_pw(_step)

## 上部の「ファイル」をクリックして一覧に戻る

プレビューが閉じ、対象ストレージのファイル一覧が表示されること。

In [ ]:
async def _step(page):
    await preview.file_list(page, target_storage_name, transition_timeout)

await run_pw(_step)

## 一覧から preview.docx を開き、変換後の1ページ目を確認する

変換が完了すると PDF ビューアーに全2ページが表示されること。1ページ目に「MFR DOCX page 1」「日本語の表示確認 1ページ目」と3色の図形・英数字が表示され、欠けや文字化けがないこと。

In [ ]:
async def _step(page):
    await preview.open_file(page, 'preview.docx', render_timeout)
    await preview.expect_pdf_page(page, 1, 'MFR DOCX page 1', render_timeout)
    await expect(preview.viewer(page).locator('#pageContainer1 .textLayer')).to_contain_text('日本語の表示確認', timeout=render_timeout)

await run_pw(_step)

## preview.docx の「次のページ」を押す

2ページ目に「MFR DOCX page 2」「日本語の表示確認 2ページ目」と図形が表示されること。

In [ ]:
async def _step(page):
    await preview.viewer(page).locator('#next').click()
    await preview.expect_pdf_page(page, 2, 'MFR DOCX page 2', render_timeout)
    await expect(preview.viewer(page).locator('#pageContainer2 .textLayer')).to_contain_text('日本語の表示確認', timeout=render_timeout)

await run_pw(_step)

## 上部の「ファイル」をクリックして一覧に戻る

プレビューが閉じ、対象ストレージのファイル一覧が表示されること。

In [ ]:
async def _step(page):
    await preview.file_list(page, target_storage_name, transition_timeout)

await run_pw(_step)

## 一覧から preview.docx をもう一度開く

ページ番号欄に `1` を入力して Enter を押す。前回の表示位置にかかわらず、1ページ目を確認する。

変換が完了すると PDF ビューアーに全2ページが表示されること。1ページ目に「MFR DOCX page 1」「日本語の表示確認 1ページ目」と3色の図形・英数字が表示され、欠けや文字化けがないこと。

In [ ]:
async def _step(page):
    await preview.open_file(page, 'preview.docx', render_timeout)
    await expect(preview.viewer(page).locator('#viewer .page canvas').first).to_be_visible(timeout=render_timeout)
    await preview.viewer(page).locator('#pageNumber').fill('1')
    await preview.viewer(page).locator('#pageNumber').press('Enter')
    await preview.expect_pdf_page(page, 1, 'MFR DOCX page 1', render_timeout)
    await expect(preview.viewer(page).locator('#pageContainer1 .textLayer')).to_contain_text('日本語の表示確認', timeout=render_timeout)

await run_pw(_step)

## 上部の「ファイル」をクリックして一覧に戻る

プレビューが閉じ、対象ストレージのファイル一覧が表示されること。

In [ ]:
async def _step(page):
    await preview.file_list(page, target_storage_name, transition_timeout)

await run_pw(_step)

## 一覧から preview.pptx を開き、変換後の1ページ目を確認する

変換が完了すると PDF ビューアーに全2ページが表示されること。1ページ目に「MFR PPTX page 1」「日本語の表示確認 1ページ目」と3色の図形・英数字が表示され、欠けや文字化けがないこと。

In [ ]:
async def _step(page):
    await preview.open_file(page, 'preview.pptx', render_timeout)
    await preview.expect_pdf_page(page, 1, 'MFR PPTX page 1', render_timeout)
    await expect(preview.viewer(page).locator('#pageContainer1 .textLayer')).to_contain_text('日本語の表示確認', timeout=render_timeout)

await run_pw(_step)

## preview.pptx の「次のページ」を押す

2ページ目に「MFR PPTX page 2」「日本語の表示確認 2ページ目」と図形が表示されること。

In [ ]:
async def _step(page):
    await preview.viewer(page).locator('#next').click()
    await preview.expect_pdf_page(page, 2, 'MFR PPTX page 2', render_timeout)
    await expect(preview.viewer(page).locator('#pageContainer2 .textLayer')).to_contain_text('日本語の表示確認', timeout=render_timeout)

await run_pw(_step)

## 上部の「ファイル」をクリックして一覧に戻る

プレビューが閉じ、対象ストレージのファイル一覧が表示されること。

In [ ]:
async def _step(page):
    await preview.file_list(page, target_storage_name, transition_timeout)

await run_pw(_step)

## 一覧から preview.pptx をもう一度開く

ページ番号欄に `1` を入力して Enter を押す。前回の表示位置にかかわらず、1ページ目を確認する。

変換が完了すると PDF ビューアーに全2ページが表示されること。1ページ目に「MFR PPTX page 1」「日本語の表示確認 1ページ目」と3色の図形・英数字が表示され、欠けや文字化けがないこと。

In [ ]:
async def _step(page):
    await preview.open_file(page, 'preview.pptx', render_timeout)
    await expect(preview.viewer(page).locator('#viewer .page canvas').first).to_be_visible(timeout=render_timeout)
    await preview.viewer(page).locator('#pageNumber').fill('1')
    await preview.viewer(page).locator('#pageNumber').press('Enter')
    await preview.expect_pdf_page(page, 1, 'MFR PPTX page 1', render_timeout)
    await expect(preview.viewer(page).locator('#pageContainer1 .textLayer')).to_contain_text('日本語の表示確認', timeout=render_timeout)

await run_pw(_step)

## 上部の「ファイル」をクリックして一覧に戻る

プレビューが閉じ、対象ストレージのファイル一覧が表示されること。

In [ ]:
async def _step(page):
    await preview.file_list(page, target_storage_name, transition_timeout)

await run_pw(_step)

## 一覧から preview.csv を開く

見出しが「名称」「数量」「日付」、データが次の2行であること。日本語・負の数・日付が欠けずに表示されること。

| 名称 | 数量 | 日付 |
| --- | --- | --- |
| 東京 | 123 | 2026-01-02 |
| 大阪 | -45 | 2026-03-04 |

In [ ]:
async def _step(page):
    await preview.open_file(page, 'preview.csv', render_timeout)
    await preview.expect_table(page, [
        ['東京', '123', '2026-01-02'], ['大阪', '-45', '2026-03-04'],
    ], render_timeout)

await run_pw(_step)

## 上部の「ファイル」をクリックして一覧に戻る

プレビューが閉じ、対象ストレージのファイル一覧が表示されること。

In [ ]:
async def _step(page):
    await preview.file_list(page, target_storage_name, transition_timeout)

await run_pw(_step)

## 一覧から preview.csv をもう一度開く

見出しが「名称」「数量」「日付」、データが次の2行であること。日本語・負の数・日付が欠けずに表示されること。

| 名称 | 数量 | 日付 |
| --- | --- | --- |
| 東京 | 123 | 2026-01-02 |
| 大阪 | -45 | 2026-03-04 |

In [ ]:
async def _step(page):
    await preview.open_file(page, 'preview.csv', render_timeout)
    await preview.expect_table(page, [
        ['東京', '123', '2026-01-02'], ['大阪', '-45', '2026-03-04'],
    ], render_timeout)

await run_pw(_step)

## 上部の「ファイル」をクリックして一覧に戻る

プレビューが閉じ、対象ストレージのファイル一覧が表示されること。

In [ ]:
async def _step(page):
    await preview.file_list(page, target_storage_name, transition_timeout)

await run_pw(_step)

## 一覧から preview.xlsx を開き、「測定値」を確認する

見出しが「名称」「数量」「日付」、データが次の2行であること。日本語・負の数・日付が欠けずに表示されること。

| 名称 | 数量 | 日付 |
| --- | --- | --- |
| 東京 | 123 | 2026-01-02 |
| 大阪 | -45 | 2026-03-04 |

「測定値」「集計」の2つのシートが表示され、最初は「測定値」が開くこと。日付型のセルは `2026-01-02T00:00:00` のように時刻付きで表示される。Excel の内部シリアル値になっていないこと。

In [ ]:
async def _step(page):
    await preview.open_file(page, 'preview.xlsx', render_timeout)
    await expect(preview.viewer(page).locator('#tabular-tabs a[role=tab]')).to_have_text(['測定値', '集計'], timeout=render_timeout)
    await preview.expect_table(page, [
        ['東京', '123', '2026-01-02T00:00:00'],
        ['大阪', '-45', '2026-03-04T00:00:00'],
    ], render_timeout)

await run_pw(_step)

## XLSX の「集計」シートをクリックする

表が切り替わり、1行だけ「合計」「78」「2026-03-05T00:00:00」が表示されること。

In [ ]:
async def _step(page):
    await preview.viewer(page).locator('#tabular-tabs').get_by_role('tab', name='集計', exact=True).click()
    await preview.expect_table(page, [['合計', '78', '2026-03-05T00:00:00']], render_timeout)

await run_pw(_step)

## XLSX の「測定値」シートに戻る

見出しが「名称」「数量」「日付」、データが次の2行であること。日本語・負の数・日付が欠けずに表示されること。

| 名称 | 数量 | 日付 |
| --- | --- | --- |
| 東京 | 123 | 2026-01-02 |
| 大阪 | -45 | 2026-03-04 |

「測定値」「集計」の2つのシートが表示され、最初は「測定値」が開くこと。日付型のセルは `2026-01-02T00:00:00` のように時刻付きで表示される。Excel の内部シリアル値になっていないこと。

In [ ]:
async def _step(page):
    await preview.viewer(page).locator('#tabular-tabs').get_by_role('tab', name='測定値', exact=True).click()
    await expect(preview.viewer(page).locator('#tabular-tabs a[role=tab]')).to_have_text(['測定値', '集計'], timeout=render_timeout)
    await preview.expect_table(page, [
        ['東京', '123', '2026-01-02T00:00:00'],
        ['大阪', '-45', '2026-03-04T00:00:00'],
    ], render_timeout)

await run_pw(_step)

## 上部の「ファイル」をクリックして一覧に戻る

プレビューが閉じ、対象ストレージのファイル一覧が表示されること。

In [ ]:
async def _step(page):
    await preview.file_list(page, target_storage_name, transition_timeout)

await run_pw(_step)

## 一覧から preview.xlsx をもう一度開く

見出しが「名称」「数量」「日付」、データが次の2行であること。日本語・負の数・日付が欠けずに表示されること。

| 名称 | 数量 | 日付 |
| --- | --- | --- |
| 東京 | 123 | 2026-01-02 |
| 大阪 | -45 | 2026-03-04 |

「測定値」「集計」の2つのシートが表示され、最初は「測定値」が開くこと。日付型のセルは `2026-01-02T00:00:00` のように時刻付きで表示される。Excel の内部シリアル値になっていないこと。

In [ ]:
async def _step(page):
    await preview.open_file(page, 'preview.xlsx', render_timeout)
    await expect(preview.viewer(page).locator('#tabular-tabs a[role=tab]')).to_have_text(['測定値', '集計'], timeout=render_timeout)
    await preview.expect_table(page, [
        ['東京', '123', '2026-01-02T00:00:00'],
        ['大阪', '-45', '2026-03-04T00:00:00'],
    ], render_timeout)

await run_pw(_step)

## 上部の「ファイル」をクリックして一覧に戻る

プレビューが閉じ、対象ストレージのファイル一覧が表示されること。

In [ ]:
async def _step(page):
    await preview.file_list(page, target_storage_name, transition_timeout)

await run_pw(_step)

## 一覧から preview.webm を開く

動画プレーヤーと再生ボタンが表示され、停止した状態で開くこと。この動画は12秒・音声なしのカラーパターンである。

In [ ]:
async def _step(page):
    await preview.open_file(page, 'preview.webm', render_timeout)
    await expect(preview.viewer(page).locator('video')).to_be_visible(timeout=render_timeout)
    await expect(preview.viewer(page).locator('video')).to_have_js_property('paused', True)

await run_pw(_step)

## 動画プレーヤーにフォーカスし、Space キーで再生する

Tab キーで動画プレーヤーへフォーカスを移し、Space キーを押す。カラーパターンが動き、再生位置が進むこと。

In [ ]:
async def _step(page):
    await preview.viewer(page).locator('video').press('Space')
    await preview.wait_video(page, 'video => !video.paused && video.currentTime >= 1 && video.videoWidth === 320', render_timeout)

await run_pw(_step)

## Space キーで動画を一時停止する

動画プレーヤーにフォーカスしたまま Space キーを押す。映像と再生位置が止まること。

In [ ]:
async def _step(page):
    await preview.viewer(page).locator('video').press('Space')
    await expect(preview.viewer(page).locator('video')).to_have_js_property('paused', True)

await run_pw(_step)

## 上部の「ファイル」をクリックして一覧に戻る

プレビューが閉じ、対象ストレージのファイル一覧が表示されること。

In [ ]:
async def _step(page):
    await preview.file_list(page, target_storage_name, transition_timeout)

await run_pw(_step)

## 一覧から preview.webm をもう一度開く

動画プレーヤーと再生ボタンが表示され、停止した状態で開くこと。この動画は12秒・音声なしのカラーパターンである。

In [ ]:
async def _step(page):
    await preview.open_file(page, 'preview.webm', render_timeout)
    await expect(preview.viewer(page).locator('video')).to_be_visible(timeout=render_timeout)
    await expect(preview.viewer(page).locator('video')).to_have_js_property('paused', True)

await run_pw(_step)

## 再表示した動画を Space キーで再生する

再表示後もカラーパターンが動き、再生位置が進むこと。

In [ ]:
async def _step(page):
    await preview.viewer(page).locator('video').press('Space')
    await preview.wait_video(page, 'video => !video.paused && video.currentTime >= 1 && video.videoWidth === 320', render_timeout)

await run_pw(_step)

## 上部の「ファイル」をクリックして一覧に戻る

プレビューが閉じ、対象ストレージのファイル一覧が表示されること。

In [ ]:
async def _step(page):
    await preview.file_list(page, target_storage_name, transition_timeout)

await run_pw(_step)

## 目視確認を終えたらテスト用プロジェクトを片付ける

`delete_project=True` の場合は、この手順で作成したプロジェクトの「設定」から「プロジェクトの削除」を選び、確認文字列を入力して削除する。プロジェクトを削除した画面が表示されること。
`False` の場合はプロジェクトを残す。残したプロジェクトは、印刷や描画の目視確認後に同じ操作で削除できる。

In [ ]:
async def _step(page):
    if delete_project:
        await grdm.delete_project(page, transition_timeout=transition_timeout)
        await expect(page.locator('#projectNavFiles a')).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)

## ブラウザーを閉じて証跡を保存する

自動実行ではブラウザーを閉じ、`default_result_path` に動画・通信記録などを保存する。手動実施では、目視結果を記録してブラウザーを閉じる。

In [ ]:
await finish_pw_context()